<a href="https://colab.research.google.com/github/tiran543/Statistical-Learning-e23094/blob/main/Copy_of_data_wrangling_assignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Assignment: Building a Modular Data Sanitization & Exploration Engine

### Background
In real-world data science, 80% of the work is spent cleaning and exploring data. Most of this work is repetitive: checking for nulls, identifying outliers, and visualizing distributions. Your task is to build a **Reusable Python Class** named `DataInspector` and a supporting `PlottingMethods` class that can be imported into Google Colab to automate these tasks.

### The Objective
Develop an end-to-end tool for CSV data ingestion, advanced cleaning, feature engineering preparation, and high-level statistical visualization.

### Technical Requirements

#### 1. Data Ingestion & Sanitization
* **Colab Integration**: Implement `upload_data()` to handle local file uploads.
* **Garbage String Handling**: Automatically recognize and convert strings like `'?'`, `'n/a'`, `'NULL'`, and `' '` into actual `NaN` values.
* **Auto-Type Correction**: Force-convert columns to numeric types if the conversion does not result in an entirely null column.

#### 2. Structural Analysis & Cleaning
* **Data Summary**: Provide a method to display row/column counts, a preview of the first 20 rows, and a breakdown of numerical vs. categorical columns.
* **Intelligent Imputation**: Create a `handle_missing_values()` method supporting multiple strategies: `mean`, `median`, `mode`, or a `constant` value.
* **Duplicate & Outlier Management**:
    * Implement `remove_duplicates()` to prune exact row matches.
    * Develop an **IQR-based** outlier detection system (`handle_outliers`) that allows users to flag or automatically delete rows based on specific columns.
* **Targeted Deletion**: Implement interactive methods (`delete_rows`, `delete_columns`) that accept comma-separated user input to prune the dataset.

#### 3. Feature Engineering Preparation (Normalization)
* **Numeric Scaling**: Implement `extract_normalized_numeric_data()` supporting `minmax`, `standard` (Z-score), and `robust` (IQR-based) scaling.
* **Categorical Encoding**: Implement `extract_normalized_categorical_data()` supporting `onehot`, `ordinal`, and `uniform` (scaled 0-1) encoding.
* **Dataset Merging**: Provide a method to create a unified DataFrame containing original numeric data alongside encoded categorical data.

#### 4. Advanced Interactive Visualization (Plotly)
* **Univariate Subplots**: For numeric columns, generate a 3-panel subplot: **Horizontal Violin/Box**, **Scatter Plot** (Index vs Value), and **Histogram**.
* **Smart Relationships**: Build a `plot_relationship()` tool that detects types and chooses the correct chart:
    * **Num-Num**: Scatter with OLS Trendline.
    * **Cat-Num**: Box plot with all data points.
    * **Cat-Cat**: Grouped Bar chart.
* **Categorical Frequency**: Create bar charts displaying both raw counts and percentage labels.

#### 5. Deep Statistical Insights
* **Unified Heatmap**: Develop `plot_all_associations_heatmap()` to visualize relationships across *all* data types:
    * **Numeric-Numeric**: Pearson’s $r$.
    * **Categorical-Categorical**: Cramér’s $V$.
    * **Mixed (Num-Cat)**: Point-Biserial correlation or Eta (via ANOVA).

#### 6. Custom Modular Plotting
Implement a separate `PlottingMethods` class to handle granular chart generation (Bar, Pie, Histogram) that returns HTML-wrapped figures for flexible embedding.

### Submission Criteria
1.  **Object-Oriented Design**: All logic must be encapsulated within the `DataInspector` and `PlottingMethods` classes.
2.  **Clean Code**: Every method must include descriptive **Docstrings** and handle empty/None data gracefully.
3.  **Real-world Testing**: Demonstrate the tool using a dataset (e.g., Titanic) by performing a full flow: Upload $\rightarrow$ Impute $\rightarrow$ Normalize $\rightarrow$ Visualize Associations.

In [ ]:
!pip install "git+https://github.com/tiran543/Statistical-Learning-e23094.git#subdirectory=data-analysis-tool"

  Cloning https://github.com/tiran543/Statistical-Learning-e23094.git to /tmp/pip-req-build-pwnh2d82
  Running command git clone --filter=blob:none --quiet https://github.com/tiran543/Statistical-Learning-e23094.git /tmp/pip-req-build-pwnh2d82
  Resolved https://github.com/tiran543/Statistical-Learning-e23094.git to commit 9cb27af99b43c0b26ae54cf197f26559958c105a
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for data-analysis-tool: filename=data_analysis_tool-1.0.0-py3-none-any.whl size=9559 sha256=fed49963d98638166831bde990f0341950bbfcb370ae0296121fcb7f8929ed83
  Stored in directory: /tmp/pip-ephem-wheel-cache-t0i6977n/wheels/b2/96/7f/3e3d6d4f248d1e3479d915bd5fbbd23b181082e6303133eca3
Successfully built data-analysis-tool


In [ ]:
from data_analysis import DataInspector, PlottingMethods

In [ ]:
import pandas as pd

inspector = DataInspector()
inspector.df = pd.read_csv("https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv")
inspector._sanitise()
print(f"✅  Loaded {inspector.df.shape[0]} rows × {inspector.df.shape[1]} columns.")

✅  Loaded 891 rows × 12 columns.


In [ ]:
inspector.data_summary()

  Rows   : 891
  Columns: 12

  Numerical columns   (8): ['PassengerId', 'Survived', 'Pclass', 'Age', 'SibSp', 'Parch', 'Ticket', 'Fare']
  Categorical columns (4): ['Name', 'Sex', 'Cabin', 'Embarked']

── First 20 rows ──


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,NaN,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,NaN,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,NaN,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803.0,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450.0,8.0500,NaN,S
5,6,0,3,"Moran, Mr. James",male,NaN,0,0,330877.0,8.4583,NaN,Q
6,7,0,1,"McCarthy, Mr. Timothy J",male,54.0,0,0,17463.0,51.8625,E46,S
7,8,0,3,"Palsson, Master. Gosta Leonard",male,2.0,3,1,349909.0,21.0750,NaN,S
8,9,1,3,"Johnson, Mrs. Oscar W (Elisabeth Vilhelmina Berg)",female,27.0,0,2,347742.0,11.1333,NaN,S
9,10,1,2,"Nasser, Mrs. Nicholas (Adele Achem)",female,14.0,1,0,237736.0,30.0708,NaN,C


In [ ]:
inspector.show_missing_data()

,missing_count,missing_%
Cabin,687,77.10
Ticket,230,25.81
Age,177,19.87
Embarked,2,0.22


In [ ]:
inspector.handle_missing_values(strategy='median')

✅  Missing values handled using 'median' strategy.


In [ ]:
inspector.remove_duplicates()

✅  Removed 0 duplicate rows. (891 remaining)


In [ ]:
inspector.plot_numerical()

In [ ]:
inspector.handle_outliers(find_and_delete=True)

  PassengerId: 0 outliers detected  [range -444.00 – 1336.00]
  Survived: 0 outliers detected  [range -1.50 – 2.50]
  Pclass: 0 outliers detected  [range 0.50 – 4.50]
  Age: 66 outliers detected  [range 2.50 – 54.50]
  SibSp: 46 outliers detected  [range -1.50 – 2.50]
  Parch: 213 outliers detected  [range 0.00 – 0.00]
  Ticket: 16 outliers detected  [range -426229.25 – 808968.75]
  Fare: 116 outliers detected  [range -26.72 – 65.63]
✅  Deleted 321 outlier rows. (570 remaining)


In [ ]:
inspector.plot_categorical()

In [ ]:
inspector.plot_relationship('Pclass', 'Fare')

In [ ]:
final_df = inspector.create_normalized_data_df()
final_df.head()

✅  Numeric data scaled using 'minmax'.
✅  Categorical data encoded using 'onehot'.
✅  Merged DataFrame: 570 rows × 645 columns.


,PassengerId,Survived,Pclass,Age,SibSp,Parch,Ticket,Fare,"Name_Abbing, Mr. Anthony","Name_Abelson, Mr. Samuel",...,Cabin_E63,Cabin_E8,Cabin_F G63,Cabin_F G73,Cabin_F33,Cabin_F38,Cabin_T,Embarked_C,Embarked_Q,Embarked_S
0,0.000000,0.0,1.0,0.346939,0.5,0.0,0.598498,0.118512,0,0,...,0,0,0,0,0,0,0,0,0,1
1,0.002247,1.0,1.0,0.428571,0.0,0.0,0.598498,0.129546,0,0,...,0,0,0,0,0,0,0,0,0,1
2,0.003371,1.0,0.0,0.612245,0.5,0.0,0.287481,0.868002,0,0,...,0,0,0,0,0,0,0,0,0,1
3,0.004494,0.0,1.0,0.612245,0.0,0.0,0.947413,0.131590,0,0,...,0,0,0,0,0,0,0,0,0,1
4,0.005618,0.0,1.0,0.469388,0.0,0.0,0.839208,0.138264,0,0,...,0,0,0,0,0,0,0,0,1,0


In [ ]:
inspector.plot_numerical_correlation()

In [ ]:
inspector.plot_categorical_correlation()

In [ ]:
inspector.plot_all_associations_heatmap()

In [ ]:
PLT = PlottingMethods()
result = PLT.plot_pie_chart(names='Sex', values='Fare', data=inspector.df)
PLT.display_image(result=result)